# BnSentMix — Model 5: Clause-Aware Dual-Polarity XLM-R

## What Model 4 established

Model 4 narrowly rejected H4. Its mean validation macro-F1 was `0.7130 ± 0.0021` and mixed-F1 was `0.6258 ± 0.0054`, versus Model 2's `0.7182 ± 0.0041` and `0.6282 ± 0.0162`. The respective deltas were only `−0.0052` and `−0.0024`.

The useful mechanism result is more specific: Model 4 achieved high mixed precision (`0.7586`) but low mixed recall (`0.5326`). For reference seed 42, only 155 of 291 mixed examples activated both evidence outputs; the remaining mixed examples were decoded as positive (41), negative (49), or neutral (46). Each evidence head worked reasonably well (`positive F1=0.7871`, `negative F1=0.7845` on average), but their **joint activation** remained difficult.

Therefore, Model 5 does not revisit class weighting or thresholds. It tests whether preserving separate clause representations helps the positive and negative evidence outputs find opposing sentiment expressed in different parts of a text.

## Hypothesis H5 and literature basis

**H5:** Relative to Model 4, clause-aware, evidence-specific pooling will increase mean mixed-F1 by at least `0.005`, while mean macro-F1 remains within `0.005` of Model 4.

The [BnSentMix paper](https://aclanthology.org/2025.loreslm-1.4.pdf) defines mixed sentiment as positive and negative sentiments conveyed in different parts of a text. [Tan et al. (2019)](https://aclanthology.org/D19-1342/) show that conflict opinions benefit from explicitly modelling positive and negative evidence. [Mukherjee and Bhattacharyya (2012)](https://aclanthology.org/C12-1113.pdf) show that lightweight discourse markers such as *but* and *although* help sentiment analysis in noisy social-media text where heavy parsers are unreliable.

Model 5 uses these ideas conservatively: punctuation and a fixed Bengali-English contrast-marker list create clauses; XLM-R still encodes the complete sequence; positive and negative learned queries attend over the clause representations separately. Markers are retained in the input. No external parser or sentiment lexicon is used.

## Frozen intervention and decision rules

| Item | Fixed value |
|---|---|
| Dataset/split | Same verified duplicate-group-disjoint split |
| Backbone | `FacebookAI/xlm-roberta-base` |
| Targets/loss | Same dual-polarity targets and unweighted BCE as Model 4 |
| Evidence threshold | Fixed `0.5` for both outputs |
| Model 5 change | Clause means + separate positive/negative clause attention |
| Clause boundaries | `. ! ? , ;` and a predeclared contrast-marker list |
| Maximum clauses | 8; extra clauses are merged into the eighth |
| Maximum length | 128 tokens; balanced prefix retention across clauses |
| Batch / LR / epochs | Same as Model 4 |
| Seeds | 13, 42, 77 |
| Checkpoint metric | Validation macro-F1 |
| Final test | Sealed |

**H5 incremental rule:** mixed-F1 ≥ `0.6308` (`0.6258 + 0.005`) and macro-F1 ≥ `0.7080` (`0.7130 − 0.005`).

**Stronger central-RQ check:** Model 5 also needs mixed-F1 > `0.6282` and macro-F1 ≥ `0.7132` to outperform the strong flat Model 2 under its original non-inferiority rule. Both decisions are reported; neither is changed after training.

In [ ]:
# Imports and all experimental choices are declared before training.
from pathlib import Path
import gc
import hashlib
import inspect
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from IPython.display import display
from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModel, AutoTokenizer, Trainer, TrainerCallback,
    TrainingArguments, set_seed
)
from transformers.modeling_outputs import SequenceClassifierOutput

DATA_PATH = Path('/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv')
PROTOCOL_DIR = Path('/kaggle/working/bnsentmix_protocol_2026')
MODEL_DIR = Path('/kaggle/working/bnsentmix_model5_clause_aware')
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_RAW_SHA256 = '148f23eb3dc40c1012a973efec920eaccc39700a74e5bcfb56806b0bf389029d'
EXPECTED_SPLIT_SHA256 = '7be81e52030967319d43ddc1e1feaf9817f9c3011a9b914265c34b5ce0ab8ec5'
EXPECTED_SPLIT_COUNTS = {'train': 13862, 'validation': 2975, 'test': 2973}
SPLIT_SEED = 2026

LABELS = [0, 1, 2, 3]
LABEL_NAMES = {0: 'positive', 1: 'negative', 2: 'neutral', 3: 'mixed'}
MIXED_LABEL = 3
TARGET_MAP = {0: [1.0, 0.0], 1: [0.0, 1.0], 2: [0.0, 0.0], 3: [1.0, 1.0]}
EVIDENCE_THRESHOLD = 0.5

MODEL_NAME = 'FacebookAI/xlm-roberta-base'
MAX_LENGTH = 128
MAX_CLAUSES = 8
CONTRAST_MARKERS = (
    'but', 'kintu', 'kinto', 'tobe', 'jodio',
    'however', 'although', 'though', 'yet'
)
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.10
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
RUN_SEEDS = [13, 42, 77]
REFERENCE_SEED = 42
OPEN_FINAL_TEST = False

MODEL4_MEAN_MACRO_F1 = 0.7130
MODEL4_MEAN_MIXED_F1 = 0.6258
MODEL2_MEAN_MACRO_F1 = 0.7182
MODEL2_MEAN_MIXED_F1 = 0.6282
MIN_MIXED_GAIN = 0.005
MACRO_NONINFERIORITY_MARGIN = 0.005

USE_FP16 = torch.cuda.is_available()
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('GPU count:', GPU_COUNT)
print('Device 0:', torch.cuda.get_device_name(0) if GPU_COUNT else 'CPU')
print('Contrast markers:', CONTRAST_MARKERS)
print('Final test open:', OPEN_FINAL_TEST)

In [ ]:
# Reconstruct and verify the identical Models 1–4 split.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def normalized_group_key(text):
    text = unicodedata.normalize('NFKC', str(text)).lower()
    text = re.sub(r'[\u200b-\u200d\ufeff]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

raw = pd.read_csv(DATA_PATH)
assert sha256_file(DATA_PATH) == EXPECTED_RAW_SHA256, 'Raw dataset differs from Models 1–4.'
lookup = {column.strip().lower(): column for column in raw.columns}
text_col = next(lookup[name] for name in ['sentence', 'text', 'comment'] if name in lookup)
label_col = next(lookup[name] for name in ['label', 'sentiment', 'class'] if name in lookup)
df = raw[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}).copy()
df.insert(0, 'source_row_id', np.arange(len(df)))
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)
df = df[df['text'].ne('')].copy()
df['group_id'] = df['text'].map(normalized_group_key)
df = df.drop_duplicates(['group_id', 'label'], keep='first').reset_index(drop=True)

splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SPLIT_SEED)
fold_ids = np.full(len(df), -1, dtype=int)
for fold_id, (_, held_out) in enumerate(
    splitter.split(df['text'], df['label'], groups=df['group_id'])
):
    fold_ids[held_out] = fold_id
df['split'] = np.where(fold_ids < 14, 'train', np.where(fold_ids < 17, 'validation', 'test'))
assert df.groupby('group_id')['split'].nunique().max() == 1
assert df['split'].value_counts().to_dict() == EXPECTED_SPLIT_COUNTS

split_path = PROTOCOL_DIR / 'bnsentmix_locked_split.csv'
df.to_csv(split_path, index=False)
assert sha256_file(split_path) == EXPECTED_SPLIT_SHA256, 'Split differs from Models 1–4.'
train_df = df[df['split'].eq('train')].copy()
val_df = df[df['split'].eq('validation')].copy()
test_df = df[df['split'].eq('test')].copy()
display(pd.crosstab(df['split'], df['label']).rename(columns=LABEL_NAMES))
print('Verified split SHA-256:', EXPECTED_SPLIT_SHA256)

In [ ]:
# Dual-polarity encoding is unchanged from Model 4.
def labels_to_evidence(labels):
    return np.asarray([TARGET_MAP[int(label)] for label in labels], dtype=np.float32)

def evidence_to_labels(evidence):
    evidence = np.asarray(evidence).astype(int)
    positive_on = evidence[:, 0] == 1
    negative_on = evidence[:, 1] == 1
    labels = np.full(len(evidence), 2, dtype=int)
    labels[positive_on & ~negative_on] = 0
    labels[~positive_on & negative_on] = 1
    labels[positive_on & negative_on] = 3
    return labels

# The marker list is fixed above and is not expanded after viewing results.
CONTRAST_REGEX = re.compile(
    r'\b(?:' + '|'.join(map(re.escape, CONTRAST_MARKERS)) + r')\b',
    flags=re.IGNORECASE,
)
PUNCTUATION_REGEX = re.compile(r'[.!?,;]+')

def split_clauses(text):
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if not text:
        return ['']

    # Punctuation stays with the preceding clause; a contrast marker starts
    # the following clause and therefore remains visible to XLM-R.
    boundaries = set()
    for match in PUNCTUATION_REGEX.finditer(text):
        if 0 < match.end() < len(text):
            boundaries.add(match.end())
    for match in CONTRAST_REGEX.finditer(text):
        if 0 < match.start() < len(text):
            boundaries.add(match.start())

    clauses, start = [], 0
    for boundary in sorted(boundaries) + [len(text)]:
        clause = text[start:boundary].strip()
        if clause:
            clauses.append(clause)
        start = boundary
    if not clauses:
        clauses = [text]
    if len(clauses) > MAX_CLAUSES:
        clauses = clauses[:MAX_CLAUSES - 1] + [' '.join(clauses[MAX_CLAUSES - 1:])]
    return clauses

# Structural audit uses training text only; the test partition remains untouched.
train_clause_audit = train_df[['label', 'text']].copy()
train_clause_audit['clause_count'] = train_clause_audit['text'].map(lambda x: len(split_clauses(x)))
train_clause_audit['has_contrast_marker'] = train_clause_audit['text'].map(
    lambda x: bool(CONTRAST_REGEX.search(str(x)))
)
clause_summary = train_clause_audit.groupby('label').agg(
    samples=('text', 'size'),
    mean_clauses=('clause_count', 'mean'),
    multi_clause_rate=('clause_count', lambda s: float((s > 1).mean())),
    contrast_marker_rate=('has_contrast_marker', 'mean'),
).rename(index=LABEL_NAMES)
display(clause_summary.round(4))

In [ ]:
# Build clause-aligned XLM-R inputs while keeping the 128-token budget.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def balanced_prefix_lengths(tokenized_clauses, budget):
    # Round-robin allocation prevents a long first clause from deleting all
    # later clauses. This matters for text such as 'good ... but bad ...'.
    allocations = [0] * len(tokenized_clauses)
    while budget > 0:
        progressed = False
        for index, token_ids in enumerate(tokenized_clauses):
            if budget == 0:
                break
            if allocations[index] < len(token_ids):
                allocations[index] += 1
                budget -= 1
                progressed = True
        if not progressed:
            break
    return allocations

def encode_clause_text(text):
    clauses = split_clauses(text)
    tokenized_clauses = [
        tokenizer.encode(clause, add_special_tokens=False) for clause in clauses
    ]
    separator_count = max(len(clauses) - 1, 0)
    token_budget = MAX_LENGTH - 2 - separator_count
    allocations = balanced_prefix_lengths(tokenized_clauses, token_budget)

    input_ids = [tokenizer.cls_token_id]
    clause_ids = [-1]
    for clause_index, (token_ids, keep_count) in enumerate(
        zip(tokenized_clauses, allocations)
    ):
        if clause_index > 0:
            input_ids.append(tokenizer.sep_token_id)
            clause_ids.append(-1)
        input_ids.extend(token_ids[:keep_count])
        clause_ids.extend([clause_index] * keep_count)
    input_ids.append(tokenizer.sep_token_id)
    clause_ids.append(-1)

    assert len(input_ids) <= MAX_LENGTH
    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
        'clause_ids': clause_ids,
    }

class ClauseAwareDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['text'].tolist()
        self.targets = labels_to_evidence(frame['label']).tolist()

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        encoded = encode_clause_text(self.texts[index])
        encoded['labels'] = self.targets[index]
        return encoded

class ClauseDataCollator:
    def __call__(self, features):
        features = [dict(feature) for feature in features]
        labels = [feature.pop('labels') for feature in features]
        clause_ids = [feature.pop('clause_ids') for feature in features]
        batch = tokenizer.pad(
            features, padding=True,
            pad_to_multiple_of=8 if USE_FP16 else None,
            return_tensors='pt',
        )
        padded_length = batch['input_ids'].shape[1]
        batch['clause_ids'] = torch.tensor([
            ids + [-1] * (padded_length - len(ids)) for ids in clause_ids
        ], dtype=torch.long)
        batch['labels'] = torch.tensor(labels, dtype=torch.float32)
        return batch

train_dataset = ClauseAwareDataset(train_df)
val_dataset = ClauseAwareDataset(val_df)
data_collator = ClauseDataCollator()

VISIBLE_DEVICES = max(GPU_COUNT, 1)
GLOBAL_EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * VISIBLE_DEVICES * GRADIENT_ACCUMULATION_STEPS
BATCHES_PER_EPOCH = math.ceil(len(train_dataset) / (TRAIN_BATCH_SIZE * VISIBLE_DEVICES))
UPDATES_PER_EPOCH = math.ceil(BATCHES_PER_EPOCH / GRADIENT_ACCUMULATION_STEPS)
TOTAL_UPDATE_STEPS = UPDATES_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = math.ceil(TOTAL_UPDATE_STEPS * WARMUP_FRACTION)
print('Training / validation:', len(train_dataset), '/', len(val_dataset))
print('Global effective batch:', GLOBAL_EFFECTIVE_BATCH_SIZE)
print('Maximum optimizer steps / warmup:', TOTAL_UPDATE_STEPS, '/', WARMUP_STEPS)

In [ ]:
# Four-class and evidence metrics remain identical to Model 4.
def sigmoid_numpy(logits):
    logits = np.clip(np.asarray(logits), -50, 50)
    return 1.0 / (1.0 + np.exp(-logits))

def predictions_from_logits(logits):
    probabilities = sigmoid_numpy(logits)
    evidence_predictions = (probabilities >= EVIDENCE_THRESHOLD).astype(int)
    return probabilities, evidence_predictions, evidence_to_labels(evidence_predictions)

def calculate_metrics(true_evidence, predicted_evidence):
    true_evidence = np.asarray(true_evidence).astype(int)
    predicted_evidence = np.asarray(predicted_evidence).astype(int)
    labels = evidence_to_labels(true_evidence)
    predictions = evidence_to_labels(predicted_evidence)
    mixed_precision, mixed_recall, mixed_f1, _ = precision_recall_fscore_support(
        labels == MIXED_LABEL, predictions == MIXED_LABEL,
        average='binary', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'macro_f1': float(f1_score(
            labels, predictions, labels=LABELS, average='macro', zero_division=0
        )),
        'weighted_f1': float(f1_score(
            labels, predictions, labels=LABELS, average='weighted', zero_division=0
        )),
        'mixed_precision': float(mixed_precision),
        'mixed_recall': float(mixed_recall),
        'mixed_f1': float(mixed_f1),
        'positive_evidence_f1': float(f1_score(
            true_evidence[:, 0], predicted_evidence[:, 0], zero_division=0
        )),
        'negative_evidence_f1': float(f1_score(
            true_evidence[:, 1], predicted_evidence[:, 1], zero_division=0
        )),
        'evidence_exact_match': float(np.mean(np.all(
            true_evidence == predicted_evidence, axis=1
        ))),
    }

def compute_metrics(eval_prediction):
    logits, true_evidence = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    _, predicted_evidence, _ = predictions_from_logits(logits)
    return calculate_metrics(true_evidence, predicted_evidence)

In [ ]:
# Model 5 intervention: evidence-specific attention over contextual clause means.
class ClauseAwareDualPolarityModel(nn.Module):
    def __init__(self, model_name, max_clauses):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.config = self.encoder.config
        self.config.num_labels = 2
        self.config.problem_type = 'multi_label_classification'
        self.max_clauses = int(max_clauses)
        hidden_size = self.config.hidden_size
        dropout_rate = getattr(self.config, 'classifier_dropout', None)
        if dropout_rate is None:
            dropout_rate = self.config.hidden_dropout_prob
        self.dropout = nn.Dropout(dropout_rate)

        # Each learned query can select a different clause. The two small
        # linear heads then combine global and evidence-specific context.
        self.positive_query = nn.Parameter(torch.empty(hidden_size))
        self.negative_query = nn.Parameter(torch.empty(hidden_size))
        self.positive_head = nn.Linear(hidden_size * 2, 1)
        self.negative_head = nn.Linear(hidden_size * 2, 1)
        self._initialize_new_parameters()

    def _initialize_new_parameters(self):
        std = getattr(self.config, 'initializer_range', 0.02)
        nn.init.normal_(self.positive_query, mean=0.0, std=std)
        nn.init.normal_(self.negative_query, mean=0.0, std=std)
        for head in (self.positive_head, self.negative_head):
            nn.init.normal_(head.weight, mean=0.0, std=std)
            nn.init.zeros_(head.bias)

    @staticmethod
    def _query_pool(clause_vectors, valid_clauses, query):
        scores = torch.einsum('bkh,h->bk', clause_vectors, query)
        scores = scores / math.sqrt(clause_vectors.shape[-1])
        scores = scores.masked_fill(~valid_clauses, -1e4)
        weights = torch.softmax(scores, dim=1) * valid_clauses.to(scores.dtype)
        weights = weights / weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
        return torch.sum(clause_vectors * weights.unsqueeze(-1), dim=1)

    def forward(
        self, input_ids=None, attention_mask=None, clause_ids=None,
        labels=None, **kwargs
    ):
        encoder_output = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask, **kwargs
        )
        token_vectors = encoder_output.last_hidden_state
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

        clause_vectors, valid_clauses = [], []
        for clause_index in range(self.max_clauses):
            mask = (clause_ids == clause_index) & attention_mask.bool()
            counts = mask.sum(dim=1, keepdim=True)
            pooled = torch.sum(token_vectors * mask.unsqueeze(-1), dim=1)
            pooled = pooled / counts.clamp_min(1).to(token_vectors.dtype)
            clause_vectors.append(pooled)
            valid_clauses.append(counts.squeeze(1) > 0)
        clause_vectors = torch.stack(clause_vectors, dim=1)
        valid_clauses = torch.stack(valid_clauses, dim=1)

        positive_context = self._query_pool(
            clause_vectors, valid_clauses, self.positive_query
        )
        negative_context = self._query_pool(
            clause_vectors, valid_clauses, self.negative_query
        )
        global_context = token_vectors[:, 0, :]  # same XLM-R start-token summary
        positive_logit = self.positive_head(self.dropout(torch.cat(
            [global_context, positive_context], dim=1
        )))
        negative_logit = self.negative_head(self.dropout(torch.cat(
            [global_context, negative_context], dim=1
        )))
        logits = torch.cat([positive_logit, negative_logit], dim=1)
        return SequenceClassifierOutput(logits=logits)

# Parameter count is recorded for the paper's efficiency discussion.
preview_model = ClauseAwareDualPolarityModel(MODEL_NAME, MAX_CLAUSES)
total_parameters = sum(parameter.numel() for parameter in preview_model.parameters())
print(f'Total parameters: {total_parameters:,}')
del preview_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Loss, checkpoint selection, and strict restoration match Model 4.
class DualPolarityTrainer(Trainer):
    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        targets = inputs.pop('labels').float()
        outputs = model(**inputs)
        loss = F.binary_cross_entropy_with_logits(outputs.logits, targets)
        return (loss, outputs) if return_outputs else loss

class ExactBestStateCallback(TrainerCallback):
    def __init__(self, state_path, patience):
        self.state_path = Path(state_path)
        self.patience = int(patience)
        self.best_score = -np.inf
        self.best_epoch = None
        self.best_loss = None
        self.bad_epochs = 0

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        score = metrics.get('eval_macro_f1') if metrics else None
        if score is None:
            return control
        if score > self.best_score:
            self.best_score = float(score)
            self.best_epoch = float(state.epoch)
            self.best_loss = float(metrics['eval_loss'])
            self.bad_epochs = 0
            unwrapped_model = model.module if hasattr(model, 'module') else model
            cpu_state = {
                key: value.detach().cpu()
                for key, value in unwrapped_model.state_dict().items()
            }
            torch.save({'model_state_dict': cpu_state}, self.state_path)
        else:
            self.bad_epochs += 1
            if self.bad_epochs >= self.patience:
                control.should_training_stop = True
        return control

def load_exact_state(path):
    load_kwargs = {'map_location': 'cpu'}
    if 'weights_only' in inspect.signature(torch.load).parameters:
        load_kwargs['weights_only'] = True
    return torch.load(path, **load_kwargs)['model_state_dict']

In [ ]:
# The optimizer and training budget are controlled against Model 4.
def make_training_arguments(seed, output_dir):
    arguments = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=MAX_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type='linear',
        max_grad_norm=1.0,
        save_strategy='no',
        logging_strategy='epoch',
        load_best_model_at_end=False,
        seed=seed,
        data_seed=seed,
        fp16=USE_FP16,
        dataloader_num_workers=2,
        report_to=[],
    )
    argument_names = inspect.signature(TrainingArguments.__init__).parameters
    arguments['eval_strategy' if 'eval_strategy' in argument_names else 'evaluation_strategy'] = 'epoch'
    return TrainingArguments(**arguments)

def build_fresh_model():
    return ClauseAwareDualPolarityModel(MODEL_NAME, MAX_CLAUSES)

In [ ]:
# Train all predeclared seeds and evaluate only on validation.
def train_one_seed(seed):
    seed_dir = MODEL_DIR / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    best_state_path = seed_dir / 'best_state.pt'
    set_seed(seed)

    model = build_fresh_model()
    callback = ExactBestStateCallback(best_state_path, EARLY_STOPPING_PATIENCE)
    trainer = DualPolarityTrainer(
        model=model, args=make_training_arguments(seed, seed_dir),
        train_dataset=train_dataset, eval_dataset=val_dataset,
        data_collator=data_collator, compute_metrics=compute_metrics,
        callbacks=[callback],
    )
    trainer.train()

    if not best_state_path.exists():
        raise RuntimeError('No validation-selected state was saved.')
    model.load_state_dict(load_exact_state(best_state_path), strict=True)
    output = trainer.predict(val_dataset)
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    probabilities, evidence_predictions, class_predictions = predictions_from_logits(logits)
    true_evidence = labels_to_evidence(val_df['label'])
    metrics = calculate_metrics(true_evidence, evidence_predictions)
    metrics.update({
        'seed': seed, 'best_epoch': callback.best_epoch,
        'validation_loss': callback.best_loss,
        'best_state_path': str(best_state_path),
    })

    prediction_frame = val_df[['source_row_id', 'group_id', 'label']].copy()
    prediction_frame['positive_probability'] = probabilities[:, 0]
    prediction_frame['negative_probability'] = probabilities[:, 1]
    prediction_frame['positive_evidence_prediction'] = evidence_predictions[:, 0]
    prediction_frame['negative_evidence_prediction'] = evidence_predictions[:, 1]
    prediction_frame['prediction'] = class_predictions
    prediction_frame.to_csv(seed_dir / 'validation_predictions.csv', index=False)
    pd.DataFrame(trainer.state.log_history).to_csv(
        seed_dir / 'training_log.csv', index=False
    )

    del trainer, model, output
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics, probabilities, evidence_predictions, class_predictions

validation_rows, validation_outputs = [], {}
for run_seed in RUN_SEEDS:
    print(f'\n===== Training seed {run_seed} =====')
    metrics, probabilities, evidence_predictions, class_predictions = train_one_seed(run_seed)
    validation_rows.append(metrics)
    validation_outputs[run_seed] = {
        'probabilities': probabilities,
        'evidence_predictions': evidence_predictions,
        'class_predictions': class_predictions,
    }

validation_results = pd.DataFrame(validation_rows).sort_values('seed').reset_index(drop=True)
metric_columns = [
    'accuracy', 'macro_f1', 'weighted_f1',
    'mixed_precision', 'mixed_recall', 'mixed_f1',
    'positive_evidence_f1', 'negative_evidence_f1', 'evidence_exact_match'
]
validation_summary = validation_results[metric_columns].agg(['mean', 'std']).T
validation_results.to_csv(MODEL_DIR / 'validation_seed_results.csv', index=False)
validation_summary.to_csv(MODEL_DIR / 'validation_mean_std.csv')
tokenizer.save_pretrained(MODEL_DIR / 'tokenizer')
display(validation_results.drop(columns='best_state_path').round(4))
display(validation_summary.round(4))

In [ ]:
# Controlled comparison and both predeclared decisions.
prior_results = pd.DataFrame({
    'seed': [13, 42, 77],
    'model2_macro_f1': [0.7158, 0.7159, 0.7229],
    'model2_mixed_f1': [0.6168, 0.6211, 0.6467],
    'model4_macro_f1': [0.7140, 0.7144, 0.7106],
    'model4_mixed_f1': [0.6316, 0.6250, 0.6210],
})
paired = prior_results.merge(validation_results, on='seed', validate='one_to_one')
paired['delta_macro_vs_model4'] = paired['macro_f1'] - paired['model4_macro_f1']
paired['delta_mixed_vs_model4'] = paired['mixed_f1'] - paired['model4_mixed_f1']
paired['delta_macro_vs_model2'] = paired['macro_f1'] - paired['model2_macro_f1']
paired['delta_mixed_vs_model2'] = paired['mixed_f1'] - paired['model2_mixed_f1']
display(paired[[
    'seed', 'model4_macro_f1', 'macro_f1', 'delta_macro_vs_model4',
    'model4_mixed_f1', 'mixed_f1', 'delta_mixed_vs_model4'
]].round(4))
paired.to_csv(MODEL_DIR / 'model2_model4_model5_paired_validation.csv', index=False)

mean_macro = float(validation_results['macro_f1'].mean())
mean_mixed = float(validation_results['mixed_f1'].mean())
h5_increment_supported = (
    mean_mixed >= MODEL4_MEAN_MIXED_F1 + MIN_MIXED_GAIN and
    mean_macro >= MODEL4_MEAN_MACRO_F1 - MACRO_NONINFERIORITY_MARGIN
)
central_rq_supported = (
    mean_mixed > MODEL2_MEAN_MIXED_F1 and
    mean_macro >= MODEL2_MEAN_MACRO_F1 - MACRO_NONINFERIORITY_MARGIN
)
decision = pd.DataFrame([{
    'model5_mean_macro_f1': mean_macro,
    'delta_macro_vs_model4': mean_macro - MODEL4_MEAN_MACRO_F1,
    'model5_mean_mixed_f1': mean_mixed,
    'delta_mixed_vs_model4': mean_mixed - MODEL4_MEAN_MIXED_F1,
    'H5_increment_supported': h5_increment_supported,
    'central_RQ_supported_vs_Model2': central_rq_supported,
}])
decision.to_csv(MODEL_DIR / 'hypothesis_H5_decision.csv', index=False)
display(decision.round(4))
print('H5 incremental decision:', h5_increment_supported)
print('Central RQ decision versus flat Model 2:', central_rq_supported)

In [ ]:
# Reference-seed class report, confusion matrix, and mechanism diagnostics.
reference = validation_outputs[REFERENCE_SEED]
reference_predictions = reference['class_predictions']
reference_probabilities = reference['probabilities']
reference_evidence = reference['evidence_predictions']
true_labels = val_df['label'].to_numpy()

display(pd.DataFrame(classification_report(
    true_labels, reference_predictions, labels=LABELS,
    target_names=[LABEL_NAMES[label] for label in LABELS],
    output_dict=True, zero_division=0
)).T.round(4))
display(pd.DataFrame(
    confusion_matrix(true_labels, reference_predictions, labels=LABELS),
    index=[f'true_{LABEL_NAMES[label]}' for label in LABELS],
    columns=[f'pred_{LABEL_NAMES[label]}' for label in LABELS],
))

# Compare the exact bottleneck observed in Model 4's reference seed.
model5_reference_row = validation_results[validation_results['seed'].eq(REFERENCE_SEED)].iloc[0]
reference_comparison = pd.DataFrame([
    {'model': 'Model 4', 'mixed_precision': 0.7561,
     'mixed_recall': 0.5326, 'mixed_f1': 0.6250},
    {'model': 'Model 5', 'mixed_precision': model5_reference_row['mixed_precision'],
     'mixed_recall': model5_reference_row['mixed_recall'],
     'mixed_f1': model5_reference_row['mixed_f1']},
])
display(reference_comparison.round(4))

# Report performance by the predeclared structural subsets.
val_clause_counts = val_df['text'].map(lambda x: len(split_clauses(x))).to_numpy()
val_has_contrast = val_df['text'].map(
    lambda x: bool(CONTRAST_REGEX.search(str(x)))
).to_numpy()
predicted_evidence_from_classes = labels_to_evidence(reference_predictions)
true_evidence = labels_to_evidence(true_labels)
subset_definitions = {
    'all_validation': np.ones(len(val_df), dtype=bool),
    'single_clause': val_clause_counts == 1,
    'multiple_clauses': val_clause_counts > 1,
    'explicit_contrast_marker': val_has_contrast,
}
subset_rows = []
for subset_name, mask in subset_definitions.items():
    subset_metrics = calculate_metrics(
        true_evidence[mask], predicted_evidence_from_classes[mask]
    )
    subset_rows.append({
        'subset': subset_name, 'support': int(mask.sum()),
        'mixed_support': int(np.sum(true_labels[mask] == MIXED_LABEL)),
        'macro_f1': subset_metrics['macro_f1'],
        'mixed_precision': subset_metrics['mixed_precision'],
        'mixed_recall': subset_metrics['mixed_recall'],
        'mixed_f1': subset_metrics['mixed_f1'],
    })
subset_table = pd.DataFrame(subset_rows)
subset_table.to_csv(MODEL_DIR / 'reference_seed_clause_subsets.csv', index=False)
display(subset_table.round(4))

coactivation_rows = []
for label in LABELS:
    mask = true_labels == label
    coactivation_rows.append({
        'true_class': LABEL_NAMES[label], 'support': int(mask.sum()),
        'mean_positive_probability': float(reference_probabilities[mask, 0].mean()),
        'mean_negative_probability': float(reference_probabilities[mask, 1].mean()),
        'both_heads_on_rate': float(np.mean(np.all(reference_evidence[mask] == 1, axis=1))),
    })
coactivation_table = pd.DataFrame(coactivation_rows)
coactivation_table.to_csv(MODEL_DIR / 'reference_seed_coactivation.csv', index=False)
display(coactivation_table.round(4))

In [ ]:
# Save the complete frozen experiment specification.
manifest = {
    'experiment': 'Model 5: clause-aware dual-polarity XLM-R',
    'hypothesis': 'Clause-aware evidence pooling improves mixed detection',
    'dataset_sha256': EXPECTED_RAW_SHA256,
    'split_sha256': EXPECTED_SPLIT_SHA256,
    'model_name': MODEL_NAME,
    'target_map': TARGET_MAP,
    'contrast_markers': list(CONTRAST_MARKERS),
    'max_clauses': MAX_CLAUSES,
    'max_length': MAX_LENGTH,
    'evidence_threshold': EVIDENCE_THRESHOLD,
    'minimum_mixed_f1_gain': MIN_MIXED_GAIN,
    'run_seeds': RUN_SEEDS,
    'global_effective_batch_size': GLOBAL_EFFECTIVE_BATCH_SIZE,
    'test_opened': OPEN_FINAL_TEST,
}
with open(MODEL_DIR / 'experiment_manifest.json', 'w') as handle:
    json.dump(manifest, handle, indent=2)

## Final-test gate

Keep `OPEN_FINAL_TEST=False`. Open the test partition only after Models 1–6 and all hypotheses, thresholds, augmentation rules, and selection criteria are frozen. Whether H5 succeeds or fails, Model 6 will test the separate robustness hypothesis using predeclared romanization-noise consistency training.

In [ ]:
# This block remains inactive throughout model development.
if not OPEN_FINAL_TEST:
    print('Final test remains sealed.')
else:
    test_dataset = ClauseAwareDataset(test_df)
    true_test_evidence = labels_to_evidence(test_df['label'])
    test_rows = []

    for run_seed in RUN_SEEDS:
        state_path = MODEL_DIR / f'seed_{run_seed}' / 'best_state.pt'
        if not state_path.exists():
            raise FileNotFoundError(f'No selected state for seed {run_seed}.')
        model = build_fresh_model()
        model.load_state_dict(load_exact_state(state_path), strict=True)
        inference_args = TrainingArguments(
            output_dir=str(MODEL_DIR / f'test_seed_{run_seed}'),
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            fp16=USE_FP16, dataloader_num_workers=2, report_to=[]
        )
        inference_trainer = DualPolarityTrainer(
            model=model, args=inference_args, data_collator=data_collator
        )
        output = inference_trainer.predict(test_dataset)
        logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
        probabilities, evidence_predictions, class_predictions = predictions_from_logits(logits)
        metrics = calculate_metrics(true_test_evidence, evidence_predictions)
        test_rows.append({'seed': run_seed, **metrics})

        prediction_frame = test_df[['source_row_id', 'group_id', 'label']].copy()
        prediction_frame['positive_probability'] = probabilities[:, 0]
        prediction_frame['negative_probability'] = probabilities[:, 1]
        prediction_frame['prediction'] = class_predictions
        prediction_frame.to_csv(
            MODEL_DIR / f'FINAL_test_predictions_seed_{run_seed}.csv', index=False
        )
        del inference_trainer, model, output
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_results = pd.DataFrame(test_rows)
    final_summary = final_results[metric_columns].agg(['mean', 'std']).T
    final_results.to_csv(MODEL_DIR / 'FINAL_test_seed_results.csv', index=False)
    final_summary.to_csv(MODEL_DIR / 'FINAL_test_mean_std.csv')
    display(final_results.round(4))
    display(final_summary.round(4))